In [1]:
library(MultiAssayExperiment)
library(RaggedExperiment)
library(data.table)

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, rowRanges

In [31]:
drop_sa <- fread("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/sample_data/master_drop_sample_annotation_sizeFactorFiltered_0.1.tsv")
drop_sa[, master_pid := gsub( "_", ".", nct_pid)]

load("/omics/odcf/analysis/OE0246_projects/datamaster/dataMASTER_release/object/dataMASTER_241104123708.RData")
load("/omics/odcf/analysis/OE0246_projects/datamaster/ext_data/annotations/gencode19_gns_lite.RData")

In [3]:
colnames(dataMASTER)

CharacterList of length 15
[["snv"]] WES.192P9WW.tumor WES.8ZTUHJ.metastasis ... WGS.9A77B1.metastasis
[["snv_germline"]] WES.192P9WW.tumor ... WGS.4H362B.metastasis
[["snv_whitelist"]] WES.192P9WW.tumor ... WGS.9A77B1.metastasis
[["indel"]] WES.192P9WW.tumor WES.8ZTUHJ.metastasis ... WGS.9A77B1.metastasis
[["indel_germline"]] WES.192P9WW.tumor ... WGS.4H362B.metastasis
[["indel_whitelist"]] WES.192P9WW.tumor ... WGS.9A77B1.metastasis
[["sv"]] WES.192P9WW.tumor WES.8ZTUHJ.metastasis ... WGS.9A77B1.metastasis
[["sv_germline"]] WES.192P9WW.tumor ... WGS.9A77B1.metastasis
[["cnv"]] WES.1T2KTK9.metastasis WES.B171CQ9.metastasis ... WGS.9XX771.tumor
[["cnv_germline"]] WES.192P9WW.tumor ... WGS.9A77B1.metastasis
...
<5 more elements>

In [32]:
myobj = dataMASTER[,,c("snv_germline", "indel_germline")] # class: MAE


Warning message:
“'experiments' dropped; see 'drops()'”
harmonizing input:
  removing 68030 sampleMap rows not in names(experiments)
  removing 115 colData rownames not in sampleMap 'primary'



In [33]:
# 1. Get all unique column names from the MEA
all_colnames <- unlist(colnames(myobj))

# 2. Initialize an empty character vector to store matches
matched_cols <- character()

# 3. Loop through each ID in your sa table
for (pid in drop_sa$master_pid) {
  # We use fixed = TRUE because your IDs have dots (.) 
  # which would otherwise be treated as wildcards in regex
  hits <- grep(pid, all_colnames, value = TRUE, fixed = TRUE)
  
  # Append new hits to our master list
  matched_cols <- c(matched_cols, hits)
}

# 4. Remove duplicates (in case one column matched multiple PIDs)
matched_cols <- unique(matched_cols)

cnv_germline  = getWithColData(myobj, "snv_germline")
actual_cnv_cols <- colnames(cnv_germline)

# 2. Filter your matched_cols to only include those present in CNV
valid_cnv_matches <- intersect(matched_cols, actual_cnv_cols)

# 3. Now subset safely
cnv_germline <- cnv_germline[, valid_cnv_matches, ]
cnv_germline


cnv_germline@assays[[1]]$Gene = as.character(NA)
sv = as(cnv_germline, "GRangesList")
 
n = length(sv)
sv = lapply(1:n, function(i) {
  svPID = sv[[i]]
  # cat(sprintf("\r%s/%s", i, n))
  idx = findOverlaps(svPID, gencode19_gns_lite)
  gns = tapply(idx@to, idx@from, function(i) {
    paste(gencode19_gns_lite$gene_name[i], collapse=",")
  })
  svPID$Gene[as.numeric(names(gns))] = unname(gns)
  svPID
})
sv_re = RaggedExperiment(sv, colData=colData(cnv_germline))

myobj[["snv_germline"]] = sv_re
germline_cnv_df <- as.data.frame(myobj[["snv_germline"]])
setDT(germline_cnv_df)
exploded_dt <- germline_cnv_df[, tidyr::separate_rows(.SD, Gene, sep = ",")]

fwrite(exploded_dt, "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_snv.tsv", sep="\t")



class: RaggedExperiment 
dim: 1621623 3821 
assays(29): Ref Alt ... CLNSIG CLNSIGCONF
rownames: NULL
colnames(3821): WES.00JN.metastasis WES.019E.metastasis ...
  WGS.ZZU2PY.tumor WGS.ZZXYWB.metastasis05
colData names(59): SampleID ProjectID ... INDEL_ClinFunOnTarget
  MutPerMb

In [ ]:
# 1. Get all unique column names from the MEA
all_colnames <- unlist(colnames(myobj))

# 2. Initialize an empty character vector to store matches
matched_cols <- character()

# 3. Loop through each ID in your sa table
for (pid in drop_sa$master_pid) {
  # We use fixed = TRUE because your IDs have dots (.) 
  # which would otherwise be treated as wildcards in regex
  hits <- grep(pid, all_colnames, value = TRUE, fixed = TRUE)
  
  # Append new hits to our master list
  matched_cols <- c(matched_cols, hits)
}

# 4. Remove duplicates (in case one column matched multiple PIDs)
matched_cols <- unique(matched_cols)

cnv_germline  = getWithColData(myobj, "indel_germline")
actual_cnv_cols <- colnames(cnv_germline)

# 2. Filter your matched_cols to only include those present in CNV
valid_cnv_matches <- intersect(matched_cols, actual_cnv_cols)

# 3. Now subset safely
cnv_germline <- cnv_germline[, valid_cnv_matches, ]
cnv_germline


cnv_germline@assays[[1]]$Gene = as.character(NA)
sv = as(cnv_germline, "GRangesList")
 
n = length(sv)
sv = lapply(1:n, function(i) {
  svPID = sv[[i]]
  # cat(sprintf("\r%s/%s", i, n))
  idx = findOverlaps(svPID, gencode19_gns_lite)
  gns = tapply(idx@to, idx@from, function(i) {
    paste(gencode19_gns_lite$gene_name[i], collapse=",")
  })
  svPID$Gene[as.numeric(names(gns))] = unname(gns)
  svPID
})
sv_re = RaggedExperiment(sv, colData=colData(cnv_germline))

myobj[["indel_germline"]] = sv_re
germline_cnv_df <- as.data.frame(myobj[["indel_germline"]])
setDT(germline_cnv_df)
exploded_dt <- germline_cnv_df[, tidyr::separate_rows(.SD, Gene, sep = ",")]

fwrite(exploded_dt, "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_indel.tsv", sep="\t")

